## DLinear: Are Transformers Effective for Time Series Forecasting?

## 사용 예제

### 데이터 불러오기

In [ ]:
pip install -U finance-datareader # 필요한 패키지를 설치

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| Device:', device)


In [ ]:
import FinanceDataReader as fdr
import numpy as np
import matplotlib.pyplot as plt


df = fdr.DataReader('000660') 

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# 전처리
dfy = df[['Close']] #

In [ ]:
# MinMaxScaler 객체 생성
scaler = MinMaxScaler()

# 'Close' 열을 min-max scaling
dfy['Close'] = scaler.fit_transform(dfy[['Close']])

In [ ]:
dfy

In [ ]:
X = dfy.values.tolist()

In [ ]:
window_size = 10

forecasting_size = 5

data_X = []
data_y = []
for i in range(len(X) - window_size - forecasting_size): #range가 (0,10)이라면 0번부터 9번까지 반복이됩니다. 여기선 0에서 294번까지 반복됨
    _X = X[i : i + window_size] # 처음에는 0 : 0+10으로 1일부터 10일까지의 데이터가 들어갑니다. 다음에는 1: 1+10 으로 2일부터 11일까지 데이터가 들어갑니다.
    _y = X[i + window_size:i + window_size + forecasting_size]     # 처음에는 10으로 11일째 종가가 들어갑니다. 다음에는 12일째 종가가 들어갑니다.
    data_X.append(_X)
    data_y.append(_y)

In [ ]:
# train data: 전체의 70%
train_size = int(len(data_y) * 0.7)

train_X = np.array(data_X[0 : train_size])
train_y = np.array(data_y[0 : train_size])


In [ ]:
# test data: 나머지 30%

test_X = np.array(data_X[train_size : len(data_X)])
test_y = np.array(data_y[train_size : len(data_y)])


print('훈련 데이터의 크기 :', train_X.shape, train_y.shape)
print('테스트 데이터의 크기 :', test_X.shape, test_y.shape)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class TSData(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32) ##
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.Y)


    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

In [ ]:
train_dataset = TSData(train_X, train_y)
test_dataset = TSData(test_X, test_y)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
for x,y in train_loader:
  break

print(x.shape, y.shape) ## (Batch, 10, 1), (Batch, 5, 1)

### 모델 구축

In [ ]:
class moving_avg(nn.Module):
    def __init__(self, kernel_size, stride):
        super(moving_avg, self).__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # 시계열의 양 끝에 패딩 추가하여 길이 유지
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1) // 2, 1) # 첫번째 값 반복
        end = x[:, -1:, :].repeat(1, (self.kernel_size - 1) // 2, 1) # 마지막 값 반복
        # 패딩을 추가한 시계열을 평균 풀링 적용
        x = torch.cat([front, x, end], dim=1)
        x = self.avg(x.permute(0, 2, 1))
        x = x.permute(0, 2, 1)
        return x

class series_decomp(nn.Module):
    def __init__(self, kernel_size):
        super(series_decomp, self).__init__()
        self.moving_avg = moving_avg(kernel_size, stride=1)

    def forward(self, x):
        # 이동 평균을 통해 시계열의 트렌드 성분 추출
        moving_mean = self.moving_avg(x)
        # 원본 시계열에서 트렌드 성분을 제거하여 잔차(residual) 계산
        res = x - moving_mean
        return res, moving_mean

class D_Linear(nn.Module):
    """
    DLinear 모델
    """
    def __init__(self, lag, horizon):
        super(D_Linear, self).__init__()
        self.Lag = lag  # 입력 시퀀스의 길이
        self.Horizon = horizon  # 예측 시퀀스의 길이

        # 시계열 분해에 사용될 커널 크기
        kernel_size = 25
        self.decompsition = series_decomp(kernel_size)
        # 계절성 성분에 대한 단일 선형 레이어 생성
        self.Linear_Seasonal = nn.Linear(self.Lag, self.Horizon)
        # 트렌드 성분에 대한 단일 선형 레이어 생성
        self.Linear_Trend = nn.Linear(self.Lag, self.Horizon)
        # 각 선형 레이어의 가중치 초기화
        self.Linear_Seasonal.weight = nn.Parameter((1 / self.Lag) * torch.ones([self.Horizon, self.Lag]))
        self.Linear_Trend.weight = nn.Parameter((1 / self.Lag) * torch.ones([self.Horizon, self.Lag]))

    def forward(self, x):
        # x: [배치 크기, 입력 길이, 채널]
        # 시계열 분해를 통해 계절성 성분과 트렌드 성분을 분리
        seasonal_init, trend_init = self.decompsition(x)
        # 차원 변환: [배치 크기, 입력 길이, 채널] -> [배치 크기, 채널, 입력 길이]
        seasonal_init, trend_init = seasonal_init.permute(0, 2, 1), trend_init.permute(0, 2, 1)
        # 계절성 성분과 트렌드 성분을 예측
        seasonal_output = self.Linear_Seasonal(seasonal_init) 
        trend_output = self.Linear_Trend(trend_init) 
        # 계절성 성분과 트렌드 성분을 더하여 최종 예측 결과 생성
        x = seasonal_output + trend_output
        # 차원 변환: [배치 크기, 채널, 예측 길이] -> [배치 크기, 예측 길이, 채널]
        return x.permute(0, 2, 1) 

In [ ]:
model = D_Linear(10,5)

### 학습

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model = model.to(device)
for epoch in range(1, 51):
    model.train()
    for x, y in train_loader:
      x , y = x.to(device), y.to(device)
      optimizer.zero_grad()
      pred = model(x)
      loss = criterion(pred, y)
      loss.backward()
      optimizer.step()
      if epoch % 10 == 0 or epoch == 1:
          print(f'Epoch {epoch:02d} loss={loss.item():.6f}')


### 추론

In [ ]:
model.eval()
with torch.no_grad():
  y_pred_list = []
  for x, y in test_loader:
    y_pred = model(x.to(device))
    y_pred_list.append(y_pred.cpu().numpy())

y_preds = np.concatenate(y_pred_list, axis=0)

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

def plot_forecast_grid(pred, target, context, n_show=9, figsize=(12, 12)):

    ctx = np.asarray(context).squeeze()    # (896, 10)
    pred = np.asarray(pred).squeeze()      # (896, 5)
    target = np.asarray(target).squeeze()  # (896, 5)
    lag = ctx.shape[1]       # 10
    horizon = pred.shape[1]  # 5
    x_ctx = np.arange(lag)                  # 0~9
    x_fcst = np.arange(lag, lag + horizon)  # 10~14
    indices = random.sample(range(pred.shape[0]), min(n_show, pred.shape[0]))
    fig, axes = plt.subplots(3, 3, figsize=figsize)
    for ax, idx in zip(axes.flatten(), indices):
        ax.plot(x_ctx, ctx[idx], 'b-o', label=f'Input ({lag}d)')      # 파란: 과거
        ax.plot(x_fcst, target[idx], 'g-o', label=f'Real ({horizon}d)') # 초록: 실제
        ax.plot(x_fcst, pred[idx], 'r--o', label=f'Pred ({horizon}d)') # 빨강: 예측
        ax.axvline(lag - 0.5, color='gray', linestyle=':', alpha=0.7)  # 과거|미래 구분
        ax.set_title(f'sample {idx}')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_forecast_grid(y_preds, test_y, test_X)